# Experiment: Clinical Missing Inference

## 기준
기존 best FE + Cat 단일 valid: 0.737416

## 결과
roc_auc: 0.7376029770
f1: 0.5150678051
precision: 0.3859035147
recall: 0.7741959837
bestIteration: 1014

## 개선폭
+0.000187

## FI
- DI_배아프로세스_구조적결측률: 1.524398
- DI_배아프로세스_대부분결측: 0.650342
- DI_배아프로세스_구조적결측수: 0.587240
- 배란유도_임상추정: 0.356480
- 임상근거_ICSI: 0.145129
- 배란유도_자연주기추정: 0.083401
- 임상근거_FER: 0.023299
- 특정시술유형_결측여부: 0.000000
- 특정시술유형_임상추정: 0.000000

## 판단
OOF 승격. 단, zero FI feature 2개는 제외 후보.

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score

from catboost import CatBoostClassifier

In [2]:
TRAIN_PATH = "../data/train.csv"
TEST_PATH = "../data/test.csv"
TARGET = "임신 성공 여부"

train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)

X_raw = train_raw.drop(columns=[TARGET])
y = train_raw[TARGET].astype(int)

X_test_raw = test_raw.copy()

id_cols = [col for col in X_raw.columns if "ID" in col.upper()]

X_raw = X_raw.drop(columns=id_cols, errors="ignore")
X_test_raw = X_test_raw.drop(columns=id_cols, errors="ignore")

print(X_raw.shape, y.shape)

(256351, 67) (256351,)


In [7]:
def _to_num_safe(s):
    return pd.to_numeric(s, errors="coerce")


def _is_missing_category(s):
    s_str = s.astype(str).str.strip()
    return (
        s.isna() |
        s_str.isin(["", "nan", "NaN", "None", "알 수 없음"])
    )


def add_clinical_missing_inference_features(df):
    df = df.copy()

    # ===============================
    # 1. 특정 시술 유형 결측 여부
    # ===============================
    if "특정 시술 유형" in df.columns:
        specific_missing = _is_missing_category(df["특정 시술 유형"])
    else:
        specific_missing = pd.Series(False, index=df.index)

    df["특정시술유형_결측여부"] = specific_missing.astype(int)

    # ===============================
    # 2. FER / 냉동 배아 사용 근거
    # 결측 조건과 분리해서 생성
    # ===============================
    thawed_embryo = (
        _to_num_safe(df["해동된 배아 수"]).fillna(0)
        if "해동된 배아 수" in df.columns
        else pd.Series(0, index=df.index)
    )

    embryo_thaw_day_present = (
        df["배아 해동 경과일"].notna()
        if "배아 해동 경과일" in df.columns
        else pd.Series(False, index=df.index)
    )

    frozen_embryo_used = (
        df["동결 배아 사용 여부"].fillna(0).astype(str).isin(["1", "1.0", "True", "true"])
        if "동결 배아 사용 여부" in df.columns
        else pd.Series(False, index=df.index)
    )

    df["임상근거_FER"] = (
        (thawed_embryo >= 1) |
        embryo_thaw_day_present |
        frozen_embryo_used
    ).astype(int)

    # 특정 시술 유형이 결측인 경우에만 별도 추정 flag
    df["특정시술결측_FER추정"] = (
        specific_missing &
        (df["임상근거_FER"] == 1)
    ).astype(int)

    # ===============================
    # 3. ICSI 근거
    # 결측 조건과 분리해서 생성
    # ===============================
    micro_injected_egg = (
        _to_num_safe(df["미세주입된 난자 수"]).fillna(0)
        if "미세주입된 난자 수" in df.columns
        else pd.Series(0, index=df.index)
    )

    micro_created_embryo = (
        _to_num_safe(df["미세주입에서 생성된 배아 수"]).fillna(0)
        if "미세주입에서 생성된 배아 수" in df.columns
        else pd.Series(0, index=df.index)
    )

    micro_transfer_embryo = (
        _to_num_safe(df["미세주입 배아 이식 수"]).fillna(0)
        if "미세주입 배아 이식 수" in df.columns
        else pd.Series(0, index=df.index)
    )

    df["임상근거_ICSI"] = (
        (micro_injected_egg >= 1) |
        (micro_created_embryo >= 1) |
        (micro_transfer_embryo >= 1)
    ).astype(int)

    df["특정시술결측_ICSI추정"] = (
        specific_missing &
        (df["임상근거_ICSI"] == 1)
    ).astype(int)

    # ===============================
    # 4. 특정 시술 유형 결측 임상 추정 category
    # ===============================
    df["특정시술유형_임상추정"] = "not_missing"

    df.loc[
        specific_missing,
        "특정시술유형_임상추정"
    ] = "missing_unknown"

    df.loc[
        specific_missing & (df["임상근거_FER"] == 1),
        "특정시술유형_임상추정"
    ] = "missing_inferred_FER"

    df.loc[
        specific_missing & (df["임상근거_ICSI"] == 1),
        "특정시술유형_임상추정"
    ] = "missing_inferred_ICSI"

    df.loc[
        specific_missing &
        (df["임상근거_FER"] == 1) &
        (df["임상근거_ICSI"] == 1),
        "특정시술유형_임상추정"
    ] = "missing_inferred_FER_ICSI"

    # ===============================
    # 5. 배란 유도 자연주기 추정
    # ===============================
    if "배란 유도 유형" in df.columns:
        ovulation_type_missing = _is_missing_category(df["배란 유도 유형"])
    else:
        ovulation_type_missing = pd.Series(False, index=df.index)

    if "배란 자극 여부" in df.columns:
        stimulation = df["배란 자극 여부"].astype(str).str.strip()
        no_stimulation = stimulation.isin(["0", "0.0", "False", "false", "아니오"])
    else:
        no_stimulation = pd.Series(False, index=df.index)

    df["배란유도_자연주기추정"] = (
        ovulation_type_missing &
        no_stimulation
    ).astype(int)

    df["배란유도_임상추정"] = (
        df["배란 유도 유형"].astype(str)
        if "배란 유도 유형" in df.columns
        else "unknown"
    )

    df.loc[
        df["배란유도_자연주기추정"] == 1,
        "배란유도_임상추정"
    ] = "자연주기_추정"

    # ===============================
    # 6. DI 구조적 결측
    # ===============================
    if "시술 유형" in df.columns:
        is_di = df["시술 유형"].astype(str).str.contains("DI", na=False)
    else:
        is_di = pd.Series(False, index=df.index)

    embryo_process_cols = [
        "난자 채취 경과일",
        "난자 해동 경과일",
        "난자 혼합 경과일",
        "배아 이식 경과일",
        "배아 해동 경과일",
        "총 생성 배아 수",
        "이식된 배아 수",
        "저장된 배아 수",
        "해동된 배아 수",
        "수집된 신선 난자 수",
        "혼합된 난자 수",
    ]

    existing_cols = [col for col in embryo_process_cols if col in df.columns]

    if existing_cols:
        missing_count = df[existing_cols].isna().sum(axis=1)
        missing_ratio = missing_count / len(existing_cols)

        df["DI_배아프로세스_구조적결측수"] = np.where(
            is_di,
            missing_count,
            0
        )

        df["DI_배아프로세스_구조적결측률"] = np.where(
            is_di,
            missing_ratio,
            0
        )

        df["DI_배아프로세스_대부분결측"] = (
            is_di &
            (missing_ratio >= 0.8)
        ).astype(int)
    else:
        df["DI_배아프로세스_구조적결측수"] = 0
        df["DI_배아프로세스_구조적결측률"] = 0
        df["DI_배아프로세스_대부분결측"] = 0

    return df

In [8]:
def data_preprocessing(df):
    df = df.copy()
    time_cols = [
        '임신 시도 또는 마지막 임신 경과 연수',
        '난자 해동 경과일',
        '난자 혼합 경과일',
        '배아 이식 경과일',
        '배아 해동 경과일'
    ]

    for col in time_cols:
        df[f'{col}_performed'] = (
            df[col].notnull()
        ).astype(int)


    df = df.fillna(0)
    infertility_cols = [
        '불임 원인 - 난관 질환',
        '불임 원인 - 남성 요인',
        '불임 원인 - 배란 장애',
        '불임 원인 - 여성 요인',
        '불임 원인 - 자궁경부 문제',
        '불임 원인 - 자궁내막증',
        '불임 원인 - 정자 농도',
        '불임 원인 - 정자 운동성',
        '불임 원인 - 정자 형태',
        '불임 원인 - 정자 면역학적 요인'
    ]

    male_cols = [
        '불임 원인 - 남성 요인',
        '불임 원인 - 정자 농도',
        '불임 원인 - 정자 운동성',
        '불임 원인 - 정자 형태',
        '불임 원인 - 정자 면역학적 요인'
    ]

    female_cols = [
        '불임 원인 - 난관 질환',
        '불임 원인 - 배란 장애',
        '불임 원인 - 여성 요인',
        '불임 원인 - 자궁경부 문제',
        '불임 원인 - 자궁내막증'
    ]

    df['불임원인_총개수'] = df[infertility_cols].sum(axis=1)

    df['남성_원인_수'] = df[male_cols].sum(axis=1)

    df['여성_원인_수'] = df[female_cols].sum(axis=1)

    df['남녀_복합_원인'] = (
        (df['남성_원인_수'] > 0) &
        (df['여성_원인_수'] > 0)
    ).astype(int)

    df['원인불명'] = (
        df['불임원인_총개수'] == 0
    ).astype(int)

    count_cols = [
        '총 시술 횟수',
        'IVF 시술 횟수',
        'DI 시술 횟수',
        '총 임신 횟수',
        'IVF 임신 횟수',
        'DI 임신 횟수',
        '총 출산 횟수',
        'IVF 출산 횟수',
        'DI 출산 횟수',
        '클리닉 내 총 시술 횟수'
    ]

    count_map = {
        '0회': 0,
        '1회': 1,
        '2회': 2,
        '3회': 3,
        '4회': 4,
        '5회': 5,
        '6회 이상': 6,
    }

    for col in count_cols:
        df[col] = df[col].map(count_map).astype(float)

    df['고령여부'] = df['시술 당시 나이'].isin([
        '만38-39세',
        '만40-42세',
        '만43-44세',
        '만45-50세'
    ]).astype(int)


    df['배아_생성률'] = np.where(
        df['혼합된 난자 수'] == 0,
        0,
        df['총 생성 배아 수'] / df['혼합된 난자 수']
    )

    # 2. 배아 이식 효율
    df['배아_이식률'] = np.where(
        df['총 생성 배아 수'] == 0,
        0,
        df['이식된 배아 수'] / df['총 생성 배아 수']
    )

    # 3. 배아 냉동 비율
    df['배아_냉동률'] = np.where(
        df['총 생성 배아 수'] == 0,
        0,
        df['저장된 배아 수'] / df['총 생성 배아 수']
    )
    df['IVF_임신성공률'] = np.where(
        df['IVF 시술 횟수'] == 0,
        0,
        df['IVF 임신 횟수'] / df['IVF 시술 횟수']
    )

    df['DI_임신성공률'] = np.where(
        df['DI 시술 횟수'] == 0,
        0,
        df['DI 임신 횟수'] / df['DI 시술 횟수']
    )

    df['고령_난자수_interaction'] = (
        df['고령여부'] *
        df['수집된 신선 난자 수']
    )

    df['배아이식_수행여부'] = (
        df['이식된 배아 수'] > 0
    ).astype(int)

    df['배아_이식_집중도'] = np.where(
        (df['이식된 배아 수'] + df['저장된 배아 수']) == 0,
        0,
        df['이식된 배아 수'] /
        (
            df['이식된 배아 수'] +
            df['저장된 배아 수']
        )
    )
    df["배아 생성 주요 이유"] = (
        df["배아 생성 주요 이유"]
        .astype(str)
        .astype("category")
    )
    df["특정 시술 유형"] = (
        df["특정 시술 유형"]
        .astype(str)
        .astype("category")
    )

    # 고령 × 이식 배아 수
    df['고령_배아이식'] = (
        df['고령여부'] *
        df['이식된 배아 수']
    )

    # 고령 × 총 생성 배아 수
    df['고령_배아생성'] = (
        df['고령여부'] *
        df['총 생성 배아 수']
    )

    # 고령 × 저장 배아 수
    df['고령_배아저장'] = (
        df['고령여부'] *
        df['저장된 배아 수']
    )

    # 고령 × 미세주입 난자 수
    df['고령_미세주입난자'] = (
        df['고령여부'] *
        df['미세주입된 난자 수']
    )

    df['출산_임신_전환율'] = np.where(
        df['총 임신 횟수'] == 0,
        0,
        df['총 출산 횟수'] / df['총 임신 횟수']
    )

    df['클리닉_집중도'] = np.where(
        df['총 시술 횟수'] == 0,
        0,
        df['클리닉 내 총 시술 횟수'] / df['총 시술 횟수']
    )

    df['첫_시술_여부'] = (
        df['총 시술 횟수'] == 0
    ).astype(int)



    binary_keywords = [
        "코드", "나이", "유형", "여부", "원인", "이유", "횟수", "출처"
    ]

    binary_cols = [
        col for col in df.columns
        if any(keyword in col for keyword in binary_keywords)
    ]

    df[binary_cols] = df[binary_cols].astype('category')

    object_cols = df.select_dtypes(include="object").columns

    df[object_cols] = df[object_cols].astype(str)

    cat_cols = df.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()

    for col in cat_cols:
        if col != TARGET:
            df[col] = df[col].astype(str)

    drop_cols = [
        '배아이식_수행여부'
    ]
    df = df.drop(
        columns=drop_cols
        )
    return df

In [9]:
def data_preprocessing_clinical_missing_exp(df):
    df = df.copy()

    # fillna 전에 임상 결측 유추 feature 생성
    df = add_clinical_missing_inference_features(df)

    # 기존 champion preprocessing 그대로 사용
    df = data_preprocessing(df)

    return df

In [11]:
clinical_cols = [
    "특정시술유형_결측여부",
    "임상근거_FER",
    "임상근거_ICSI",
    "특정시술결측_FER추정",
    "특정시술결측_ICSI추정",
    "특정시술유형_임상추정",
    "배란유도_자연주기추정",
    "배란유도_임상추정",
    "DI_배아프로세스_구조적결측수",
    "DI_배아프로세스_구조적결측률",
    "DI_배아프로세스_대부분결측",
]

print(X_exp[clinical_cols].nunique())
display(X_exp[clinical_cols].head())

for col in clinical_cols:
    print("\n", col)
    print(X_exp[col].value_counts(dropna=False).head(20))

특정시술유형_결측여부         2
임상근거_FER            2
임상근거_ICSI           2
특정시술결측_FER추정        1
특정시술결측_ICSI추정       1
특정시술유형_임상추정         2
배란유도_자연주기추정         2
배란유도_임상추정           5
DI_배아프로세스_구조적결측수    2
DI_배아프로세스_구조적결측률    2
DI_배아프로세스_대부분결측     2
dtype: int64


,특정시술유형_결측여부,임상근거_FER,임상근거_ICSI,특정시술결측_FER추정,특정시술결측_ICSI추정,특정시술유형_임상추정,배란유도_자연주기추정,배란유도_임상추정,DI_배아프로세스_구조적결측수,DI_배아프로세스_구조적결측률,DI_배아프로세스_대부분결측
0,0,0,1,0,0,not_missing,0,기록되지 않은 시행,0,0.0,0
1,0,0,1,0,0,not_missing,1,자연주기_추정,0,0.0,0
2,0,0,0,0,0,not_missing,0,기록되지 않은 시행,0,0.0,0
3,0,0,1,0,0,not_missing,0,기록되지 않은 시행,0,0.0,0
4,0,0,1,0,0,not_missing,0,기록되지 않은 시행,0,0.0,0



 특정시술유형_결측여부
특정시술유형_결측여부
0    256349
1         2
Name: count, dtype: int64

 임상근거_FER
임상근거_FER
0    215963
1     40388
Name: count, dtype: int64

 임상근거_ICSI
임상근거_ICSI
1    128859
0    127492
Name: count, dtype: int64

 특정시술결측_FER추정
특정시술결측_FER추정
0    256351
Name: count, dtype: int64

 특정시술결측_ICSI추정
특정시술결측_ICSI추정
0    256351
Name: count, dtype: int64

 특정시술유형_임상추정
특정시술유형_임상추정
not_missing        256349
missing_unknown         2
Name: count, dtype: int64

 배란유도_자연주기추정
배란유도_자연주기추정
0    197720
1     58631
Name: count, dtype: int64

 배란유도_임상추정
배란유도_임상추정
기록되지 않은 시행      194432
자연주기_추정          58631
알 수 없음            3286
세트로타이드 (억제제)         1
생식선 자극 호르몬           1
Name: count, dtype: int64

 DI_배아프로세스_구조적결측수
DI_배아프로세스_구조적결측수
0     250060
11      6291
Name: count, dtype: int64

 DI_배아프로세스_구조적결측률
DI_배아프로세스_구조적결측률
0.0    250060
1.0      6291
Name: count, dtype: int64

 DI_배아프로세스_대부분결측
DI_배아프로세스_대부분결측
0    250060
1      6291
Name: count, dtype: int64


In [12]:
constant_cols = [
    col for col in clinical_cols
    if X_exp[col].nunique() <= 1
]

print("constant_cols:", constant_cols)

X_exp = X_exp.drop(columns=constant_cols, errors="ignore")

constant_cols: ['특정시술결측_FER추정', '특정시술결측_ICSI추정']


In [13]:
X_train, X_val, y_train, y_val = train_test_split(
    X_exp,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape, X_val.shape)
print(y_train.mean(), y_val.mean())

(205080, 101) (51271, 101)
0.2583479617710162 0.2583526750014628


In [14]:
cat_cols = X_train.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

for col in cat_cols:
    X_train[col] = X_train[col].astype(str)
    X_val[col] = X_val[col].astype(str)

cat_model = CatBoostClassifier(
    iterations=2000,
    learning_rate=0.02498214961001344,
    depth=8,
    l2_leaf_reg=18.591182129683194,
    random_strength=0.32969640414889206,
    bagging_temperature=4.535604806522509,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=100,
    class_weights=[1, 190123 / 66228],
    allow_writing_files=False
)

cat_model.fit(
    X_train,
    y_train,
    cat_features=cat_cols,
    eval_set=(X_val, y_val),
    early_stopping_rounds=100,
    verbose=100
)

y_val_pred = cat_model.predict(X_val)
y_val_proba = cat_model.predict_proba(X_val)[:, 1]

print("f1:", f1_score(y_val, y_val_pred))
print("precision:", precision_score(y_val, y_val_pred))
print("recall:", recall_score(y_val, y_val_pred))
print("roc_auc:", roc_auc_score(y_val, y_val_proba))

0:	test: 0.7201681	best: 0.7201681 (0)	total: 2.32s	remaining: 1h 17m 17s
100:	test: 0.7340766	best: 0.7340766 (100)	total: 18.8s	remaining: 5m 53s
200:	test: 0.7364045	best: 0.7364045 (200)	total: 31.6s	remaining: 4m 43s
300:	test: 0.7370056	best: 0.7370058 (299)	total: 44s	remaining: 4m 8s
400:	test: 0.7372195	best: 0.7372205 (398)	total: 55.6s	remaining: 3m 41s
500:	test: 0.7373189	best: 0.7373255 (499)	total: 1m 7s	remaining: 3m 21s
600:	test: 0.7373991	best: 0.7374021 (597)	total: 1m 19s	remaining: 3m 5s
700:	test: 0.7374504	best: 0.7374629 (674)	total: 1m 32s	remaining: 2m 51s
800:	test: 0.7375094	best: 0.7375094 (800)	total: 1m 44s	remaining: 2m 36s
900:	test: 0.7375720	best: 0.7375734 (899)	total: 1m 56s	remaining: 2m 22s
1000:	test: 0.7375879	best: 0.7375966 (992)	total: 2m 9s	remaining: 2m 9s
1100:	test: 0.7375255	best: 0.7376030 (1014)	total: 2m 22s	remaining: 1m 56s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.737602977
bestIteration = 1014

Shrink m

In [15]:
fi = pd.DataFrame({
    "feature": X_train.columns,
    "importance": cat_model.get_feature_importance()
}).sort_values("importance", ascending=False)

display(fi.head(50))

display(
    fi[fi["feature"].isin(clinical_cols)]
)

,feature,importance
41,이식된 배아 수,25.872733
93,배아_이식_집중도,25.205006
52,난자 출처,5.956700
1,시술 당시 나이,4.804611
92,고령_난자수_interaction,3.468203
65,배아 이식 경과일,3.176439
43,저장된 배아 수,2.678651
89,배아_냉동률,2.627656
53,정자 출처,2.610036
47,수집된 신선 난자 수,2.587580


,feature,importance
74,DI_배아프로세스_구조적결측률,1.524398
75,DI_배아프로세스_대부분결측,0.650342
73,DI_배아프로세스_구조적결측수,0.587240
72,배란유도_임상추정,0.356480
69,임상근거_ICSI,0.145129
71,배란유도_자연주기추정,0.083401
68,임상근거_FER,0.023299
67,특정시술유형_결측여부,0.000000
70,특정시술유형_임상추정,0.000000
